# Auditoria de valores ausentes e limpeza inicial (PySpark)

**Bases:** `df01` SINAN Dengue 2024; `df02` Chronic Kidney Disease; `df03` MIMIC-II arterial; `df04` Periodontal Therapy; `df05` NHANES.

**Fluxo por base:** (1) leitura CSV, (2) total de linhas e colunas, (3) diagnóstico de nulos por coluna, (4) remoção de colunas com **mais de 50%** de ausências, (5) remoção dos registros que ainda tenham alguma ausência, (6) relatórios, CSV único e Parquet limpos.


> **Importante:** a primeira limpeza identifica apenas células realmente vazias, campos que contêm só espaços, `NULL`, `null`, `NaN` e outros marcadores textuais configurados abaixo. No SINAN, códigos como `9`, `99`, `999` ou `Ignorado` podem significar ausência **dependendo da variável**. Não os eliminamos globalmente, pois podem ser categorias válidas em outras colunas. Consulte o dicionário antes de fazer essa limpeza semântica.

In [1]:
from pathlib import Path
import csv
import re
import pandas as pd
from pyspark.sql import SparkSession, functions as F
import shutil

# Diretório principal
BASE_DIR = Path("/home/mirlei/datasets_rl")

# Dados originais
RAW_DIR = BASE_DIR / "0_raw"

# Dados convertidos para CSV e Parquet
CSV_DIR = BASE_DIR / "1_processed" / "csv"
PARQUET_DIR = BASE_DIR / "1_processed" / "parquet"

# Relatórios
REPORT_DIR = BASE_DIR / "reports" / "etapa_01"

# Criar diretórios
for diretorio in [CSV_DIR, PARQUET_DIR, REPORT_DIR]:
    diretorio.mkdir(parents=True, exist_ok=True)


DATASETS = {
    'df01': 'SINAN Dengue 2024',
    'df02': 'Chronic kidney disease',
    'df03': 'MIMIC-II arterial',
    'df04': 'Periodontal therapy',
    'df05': 'NHANES',
}

# Extensões de texto tratadas como ausentes (após strip e sem distinguir maiúsculas).
# Acrescente apenas marcadores textuais que comprovadamente significam ausência.
NULL_TOKENS = {'', 'null', 'none', 'nan', 'na', 'n/a'}
LIMIAR_NULOS = 0.50  # excluir SOMENTE se proporção > 0.50

spark = (SparkSession.builder
    .appName('AuditoriaDatasetsSaude_Etapa01')
    .master('local[*]')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.sql.adaptive.enabled', 'true')
    .getOrCreate())

print('Spark:', spark.version)
print('Origem:', RAW_DIR)


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/30 19:05:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 3.5.4
Origem: /home/mirlei/datasets_rl/0_raw


## 1. Verificação dos arquivos e separadores

A detecção do separador consulta somente o cabeçalho e uma pequena amostra local. Se necessário, ajuste manualmente `SEPARADORES` (por exemplo, `df01: ';'`).

In [2]:
SEPARADORES = {}  # Exemplo: {'df01': ';', 'df03': ','}


def detectar_separador(path):
    with path.open('r', encoding='utf-8-sig', errors='replace', newline='') as f:
        amostra = f.read(65536)
    try:
        return csv.Sniffer().sniff(amostra, delimiters=',;\t|').delimiter
    except csv.Error:
        primeira = amostra.splitlines()[0] if amostra else ''
        return max([',', ';', '\t', '|'], key=lambda sep: primeira.count(sep))

arquivos = []
for ident, nome in DATASETS.items():
    arquivo = RAW_DIR / f'{ident}.csv'
    if not arquivo.is_file():
        raise FileNotFoundError(f'Arquivo não encontrado: {arquivo}')
    sep = SEPARADORES.get(ident, detectar_separador(arquivo))
    arquivos.append({'id': ident, 'nome': nome, 'arquivo': str(arquivo),
                     'tamanho_MB': round(arquivo.stat().st_size / 1024**2, 2),
                     'separador': sep})
display(pd.DataFrame([{**a, 'separador': repr(a['separador'])} for a in arquivos]))

,id,nome,arquivo,tamanho_MB,separador
0,df01,SINAN Dengue 2024,/home/mirlei/datasets_rl/0_raw/df01.csv,1755.14,"','"
1,df02,Chronic kidney disease,/home/mirlei/datasets_rl/0_raw/df02.csv,0.03,"','"
2,df03,MIMIC-II arterial,/home/mirlei/datasets_rl/0_raw/df03.csv,0.28,"','"
3,df04,Periodontal therapy,/home/mirlei/datasets_rl/0_raw/df04.csv,0.07,"','"
4,df05,NHANES,/home/mirlei/datasets_rl/0_raw/df05.csv,0.11,"','"


## 2. Funções de leitura, auditoria e limpeza

- `count()` determina o total exato de linhas.
- Uma agregação Spark computa a contagem de ausências de **todas** as colunas; os dados volumosos permanecem no Spark.
- A auditoria considera espaços em branco e tokens configurados. Não faz imputação.
- A limpeza de células é feita em uma projeção única, evitando cadeias extensas de transformações.
- O CSV é salvo como **arquivo único** por dataset; o Parquet é salvo em diretório Spark.

In [3]:
def ler_csv(ident, sep):
    path = str(RAW_DIR / f'{ident}.csv')
    df = (spark.read.option('header', True)
          .option('sep', sep)
          .option('encoding', 'UTF-8')
          .option('inferSchema', False)
          .option('mode', 'FAILFAST')
          .csv(path))
    if len(df.columns) != len(set(df.columns)):
        raise ValueError(f'{ident}: nomes de colunas duplicados; renomeie antes de prosseguir.')
    if not df.columns or (len(df.columns) == 1 and sep not in (None, '')):
        print(f'AVISO: {ident} tem apenas uma coluna. Confira o separador.')
    return df


def normalizar_nulos(df):
    # Conserva todos os valores não nulos como texto (inclusive códigos categóricos).
    cols = []
    for nome in df.columns:
        valor = F.trim(F.col(f'`{nome.replace("`", "``")}`'))
        normalizado = F.when(
            valor.isNull() | F.lower(valor).isin(*sorted(NULL_TOKENS)),
            F.lit(None).cast('string')
        ).otherwise(valor)
        cols.append(normalizado.alias(nome))
    return df.select(*cols)


def contar_nulos(df):
    # Usa coluna por posição com alias seguro para suportar espaços e caracteres especiais.
    expressoes = [F.sum(F.when(F.col(f'`{c.replace("`", "``")}`').isNull(), 1).otherwise(0))
                  .cast('long').alias(f'v_{i}') for i, c in enumerate(df.columns)]
    contagens = df.agg(*expressoes).first().asDict()
    return {c: int(contagens[f'v_{i}'] or 0) for i, c in enumerate(df.columns)}


def processar(ident, nome, sep):
    print(f'\n===== {ident}: {nome} =====', flush=True)
    original = ler_csv(ident, sep)
    n_col_inicial = len(original.columns)
    df = normalizar_nulos(original)

    # Uma primeira passagem conta linhas; outra calcula ausências por coluna.
    total_linhas = df.count()
    if total_linhas == 0:
        raise ValueError(f'{ident}: arquivo sem linhas de dados')
    nulos = contar_nulos(df)
    relatorio = pd.DataFrame({
        'dataset': ident,
        'coluna': df.columns,
        'nulos': [nulos[c] for c in df.columns],
        'percentual_nulos': [round(100 * nulos[c] / total_linhas, 4) for c in df.columns],
        'remover_coluna': [nulos[c] / total_linhas > LIMIAR_NULOS for c in df.columns]
    }).sort_values(['percentual_nulos', 'coluna'], ascending=[False, True])
    relatorio.to_csv(REPORT_DIR / f'{ident}_nulos_colunas.csv', index=False)

    remover = [c for c in df.columns if nulos[c] / total_linhas > LIMIAR_NULOS]
    manter = [c for c in df.columns if c not in remover]
    if not manter:
        raise ValueError(f'{ident}: nenhuma coluna sobrou após o limiar')

    # Ordem intencional: excluir colunas >50%, DEPOIS linhas incompletas.
    filtrado = df.select(*[F.col(f'`{c.replace("`", "``")}`').alias(c) for c in manter])
    limpo = filtrado.dropna(how='any', subset=manter)
    linhas_finais = limpo.count()

    # Grava somente os registros limpos; CSV único e Parquet.
    destino_csv = CSV_DIR / f'{ident}.csv'
    destino_parquet = PARQUET_DIR / ident
    temporario = CSV_DIR / f'_temp_{ident}'
    (
        limpo.coalesce(1).write.mode('overwrite')
        .option('header', True).option('quote', '"').option('escape', '"')
        .csv(str(temporario))
    )
    arquivo_part = next(temporario.glob('part-*.csv'))
    destino_csv.unlink(missing_ok=True)
    shutil.move(str(arquivo_part), str(destino_csv))
    shutil.rmtree(temporario)
    (limpo.write.mode('overwrite')
        .option('compression', 'snappy')
        .parquet(str(destino_parquet)))

    resumo = {
        'dataset': ident, 'nome': nome,
        'registros_originais': total_linhas,
        'colunas_originais': n_col_inicial,
        'colunas_excluidas': len(remover),
        'colunas_finais': len(manter),
        'registros_excluidos': total_linhas - linhas_finais,
        'registros_finais': linhas_finais,
        'percentual_registros_excluidos': round(100 * (total_linhas - linhas_finais) / total_linhas, 4),
        'colunas_removidas': '; '.join(remover),
        'diretorio_csv_limpo': str(destino_csv),
        'diretorio_parquet_limpo': str(destino_parquet)
    }
    print('Dimensão original:', (total_linhas, n_col_inicial))
    print('Colunas >50% de nulos:', remover)
    print('Dimensão final:', (linhas_finais, len(manter)))
    print('Registros removidos:', total_linhas - linhas_finais,
          f'({resumo["percentual_registros_excluidos"]:.2f}%)', flush=True)
    print('CSV limpo salvo:', destino_csv)
    print('Parquet limpo salvo:', destino_parquet)
    return resumo, relatorio


## 3. Executar para as cinco bases

`df01` tem cerca de 1,8 GB: o tempo depende da quantidade de colunas, da CPU e do armazenamento. O processamento é **sequencial**, para limitar pressão sobre memória. A primeira execução lê os CSVs várias vezes para contar, auditar e gravar, sem carregar todo o arquivo na RAM com pandas.

In [4]:
resumos = []
relatorios = {}
for item in arquivos:
    resumo, relatorio = processar(item['id'], item['nome'], item['separador'])
    resumos.append(resumo)
    relatorios[item['id']] = relatorio

resumo_df = pd.DataFrame(resumos)
resumo_df.to_csv(REPORT_DIR / 'resumo_geral.csv', index=False)
display(resumo_df.drop(columns=['colunas_removidas', 'diretorio_csv_limpo', 'diretorio_parquet_limpo']))


===== df01: SINAN Dengue 2024 =====


26/10/30 19:06:54 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 11:====================================================>   (13 + 1) / 14]

Dimensão original: (6564924, 121)
Colunas >50% de nulos: ['ID_OCUPA_N', 'DT_CHIK_S1', 'DT_CHIK_S2', 'DT_PRNT', 'RES_CHIKS1', 'RES_CHIKS2', 'RESUL_PRNT', 'DT_SORO', 'DT_NS1', 'DT_VIRAL', 'RESUL_VI_N', 'DT_PCR', 'RESUL_PCR_', 'SOROTIPO', 'HISTOPA_N', 'IMUNOH_N', 'DT_INTERNA', 'UF', 'MUNICIPIO', 'DOENCA_TRA', 'CLINC_CHIK', 'DT_OBITO', 'ALRM_HIPOT', 'ALRM_PLAQ', 'ALRM_VOM', 'ALRM_SANG', 'ALRM_HEMAT', 'ALRM_ABDOM', 'ALRM_LETAR', 'ALRM_HEPAT', 'ALRM_LIQ', 'DT_ALRM', 'GRAV_PULSO', 'GRAV_CONV', 'GRAV_ENCH', 'GRAV_INSUF', 'GRAV_TAQUI', 'GRAV_EXTRE', 'GRAV_HIPOT', 'GRAV_HEMAT', 'GRAV_MELEN', 'GRAV_METRO', 'GRAV_SANG', 'GRAV_AST', 'GRAV_MIOC', 'GRAV_CONSC', 'GRAV_ORGAO', 'DT_GRAV', 'MANI_HEMOR', 'EPISTAXE', 'GENGIVO', 'METRO', 'PETEQUIAS', 'HEMATURA', 'SANGRAM', 'LACO_N', 'PLASMATICO', 'EVIDENCIA', 'PLAQ_MENOR', 'CON_FHD', 'COMPLICA', 'NDUPLIC_N', 'FLXRECEBI', 'MIGRADO_W']
Dimensão final: (2051634, 57)
Registros removidos: 4513290 (68.75%)
CSV limpo salvo: /home/mirlei/datasets_rl/1_processed/csv

Dimensão original: (202, 29)
Colunas >50% de nulos: []
Dimensão final: (201, 29)
Registros removidos: 1 (0.49%)
CSV limpo salvo: /home/mirlei/datasets_rl/1_processed/csv/df02.csv
Parquet limpo salvo: /home/mirlei/datasets_rl/1_processed/parquet/df02

===== df03: MIMIC-II arterial =====
Dimensão original: (1776, 46)
Colunas >50% de nulos: []
Dimensão final: (1064, 46)
Registros removidos: 712 (40.09%)
CSV limpo salvo: /home/mirlei/datasets_rl/1_processed/csv/df03.csv
Parquet limpo salvo: /home/mirlei/datasets_rl/1_processed/parquet/df03

===== df04: Periodontal therapy =====
Dimensão original: (883, 22)
Colunas >50% de nulos: []
Dimensão final: (878, 22)
Registros removidos: 5 (0.57%)
CSV limpo salvo: /home/mirlei/datasets_rl/1_processed/csv/df04.csv
Parquet limpo salvo: /home/mirlei/datasets_rl/1_processed/parquet/df04

===== df05: NHANES =====
Dimensão original: (2278, 10)
Colunas >50% de nulos: []
Dimensão final: (2278, 10)
Registros removidos: 0 (0.00%)
CSV limpo salvo: /home/mirlei

,dataset,nome,registros_originais,colunas_originais,colunas_excluidas,colunas_finais,registros_excluidos,registros_finais,percentual_registros_excluidos
0,df01,SINAN Dengue 2024,6564924,121,64,57,4513290,2051634,68.7485
1,df02,Chronic kidney disease,202,29,0,29,1,201,0.4950
2,df03,MIMIC-II arterial,1776,46,0,46,712,1064,40.0901
3,df04,Periodontal therapy,883,22,0,22,5,878,0.5663
4,df05,NHANES,2278,10,0,10,0,2278,0.0000


## 4. Inspecionar colunas com nulos

Escolha a base desejada e visualize a contagem de nulos, porcentagem e decisão de exclusão.

In [5]:
BASE_INSPECAO = 'df01'  # altere para df02, df03, df04 ou df05
display(relatorios[BASE_INSPECAO])
print('Colunas excluídas:', resumo_df.set_index('dataset').loc[BASE_INSPECAO, 'colunas_removidas'])

,dataset,coluna,nulos,percentual_nulos,remover_coluna
45,df01,DT_CHIK_S2,6564901,99.9996,True
46,df01,DT_PRNT,6564511,99.9937,True
44,df01,DT_CHIK_S1,6562063,99.9564,True
74,df01,DT_OBITO,6556934,99.8783,True
101,df01,DT_GRAV,6554652,99.8435,True
...,...,...,...,...,...
3,df01,SEM_NOT,0,0.0000,False
10,df01,SEM_PRI,0,0.0000,False
17,df01,SG_UF,0,0.0000,False
5,df01,SG_UF_NOT,0,0.0000,False


Colunas excluídas: ID_OCUPA_N; DT_CHIK_S1; DT_CHIK_S2; DT_PRNT; RES_CHIKS1; RES_CHIKS2; RESUL_PRNT; DT_SORO; DT_NS1; DT_VIRAL; RESUL_VI_N; DT_PCR; RESUL_PCR_; SOROTIPO; HISTOPA_N; IMUNOH_N; DT_INTERNA; UF; MUNICIPIO; DOENCA_TRA; CLINC_CHIK; DT_OBITO; ALRM_HIPOT; ALRM_PLAQ; ALRM_VOM; ALRM_SANG; ALRM_HEMAT; ALRM_ABDOM; ALRM_LETAR; ALRM_HEPAT; ALRM_LIQ; DT_ALRM; GRAV_PULSO; GRAV_CONV; GRAV_ENCH; GRAV_INSUF; GRAV_TAQUI; GRAV_EXTRE; GRAV_HIPOT; GRAV_HEMAT; GRAV_MELEN; GRAV_METRO; GRAV_SANG; GRAV_AST; GRAV_MIOC; GRAV_CONSC; GRAV_ORGAO; DT_GRAV; MANI_HEMOR; EPISTAXE; GENGIVO; METRO; PETEQUIAS; HEMATURA; SANGRAM; LACO_N; PLASMATICO; EVIDENCIA; PLAQ_MENOR; CON_FHD; COMPLICA; NDUPLIC_N; FLXRECEBI; MIGRADO_W


## 5. Verificar arquivos limpos salvos

A gravação dos dados limpos já acontece na função `processar()` (célula 5). Não leia novamente os CSVs originais para salvar.


In [6]:
# Os arquivos já foram salvos durante o processamento da célula 7.
print("CSV:", CSV_DIR)
print("Parquet:", PARQUET_DIR)


CSV: /home/mirlei/datasets_rl/1_processed/csv
Parquet: /home/mirlei/datasets_rl/1_processed/parquet


In [7]:
# Não é necessário repetir a gravação.


In [8]:
for ident in DATASETS:

    df_csv = spark.read.option("header", True).csv(
        str(CSV_DIR / f"{ident}.csv")
    )

    df_parquet = spark.read.parquet(
        str(PARQUET_DIR / ident)
    )

    n_csv = df_csv.count()
    n_parquet = df_parquet.count()

    col_csv = len(df_csv.columns)
    col_parquet = len(df_parquet.columns)

    status = (
        "OK"
        if n_csv == n_parquet and col_csv == col_parquet
        else "DIVERGÊNCIA"
    )

    print(
        f"{ident}: CSV={n_csv:,} linhas, {col_csv} colunas | "
        f"Parquet={n_parquet:,} linhas, {col_parquet} colunas | "
        f"{status}"
    )

df01: CSV=2,051,634 linhas, 57 colunas | Parquet=2,051,634 linhas, 57 colunas | OK
df02: CSV=201 linhas, 29 colunas | Parquet=201 linhas, 29 colunas | OK
df03: CSV=1,064 linhas, 46 colunas | Parquet=1,064 linhas, 46 colunas | OK
df04: CSV=878 linhas, 22 colunas | Parquet=878 linhas, 22 colunas | OK
df05: CSV=2,278 linhas, 10 colunas | Parquet=2,278 linhas, 10 colunas | OK
